# AI Data Analyst: Excel and edge-case validation

This notebook validates the project against the Excel sample placed in `data_sample/`. It also checks CSV ingestion and analysis behavior for common edge cases.

Before running: `uv sync --extra notebook`

In [1]:
from pathlib import Path
import sys

# Make the source package importable when Jupyter is started from the repository root.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from data_analyst.config import OUTPUT_DIR
from data_analyst.crew import run_analysis
from data_analyst.tools.data_tools import load_csv, profile_dataframe

SAMPLE_DIR = ROOT / 'data_sample'
SAMPLE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Repository: {ROOT}')
print(f'Excel sample directory: {SAMPLE_DIR}')

Repository: D:\data-analyst-agent
Excel sample directory: D:\data-analyst-agent\data_sample


In [2]:
import pandas as pd

excel_files = sorted([*SAMPLE_DIR.glob('*.xlsx'), *SAMPLE_DIR.glob('*.xls')])
if not excel_files:
    raise FileNotFoundError(
        'No Excel file found. Place the supplied .xlsx file directly in data_sample/ and rerun this cell.'
    )

excel_path = excel_files[0]
workbook = pd.ExcelFile(excel_path)
print(f'Using: {excel_path.name}')
print('Sheets:', workbook.sheet_names)

# Use the first sheet by default. Change this if a different source sheet is needed.
SHEET_NAME = workbook.sheet_names[0]
source_df = pd.read_excel(excel_path, sheet_name=SHEET_NAME)
print(f'Sheet: {SHEET_NAME} | shape: {source_df.shape}')
display(source_df.head())

Using: data_analysis_agent_test.xlsx
Sheets: ['Sales Data', 'Test Questions']
Sheet: Sales Data | shape: (150, 15)


,Order_ID,Order_Date,Region,City,Product,Category,Customer_Segment,Channel,Units,Unit_Price,Revenue,Discount_Pct,Cost,Profit,Profit_Margin_Pct
0,ORD-0001,2026-02-27,North,Chandigarh,Keyboard,Accessories,Consumer,Online,3,2500,7500,0.00,4500,3000,40.00
1,ORD-0002,2026-02-14,East,Kolkata,Laptop,Electronics,Consumer,Online,4,65000,208000,0.20,192000,16000,7.69
2,ORD-0003,2026-01-14,South,Hyderabad,Office Chair,Furniture,Enterprise,Partner,14,12000,159600,0.05,112000,47600,29.82
3,ORD-0004,2026-08-18,West,Mumbai,Desk,Furniture,Consumer,Partner,7,18000,113400,0.10,87500,25900,22.84
4,ORD-0005,2026-05-23,South,Bengaluru,Desk,Furniture,SMB,Online,2,18000,30600,0.15,25000,5600,18.30


In [3]:
# Dataset profile used by the API's local analyst.
profile = profile_dataframe(source_df)
print(f"Rows: {profile['rows']:,}")
print(f"Columns: {profile['columns']}")
print(f"Missing cells: {profile['missing_cells']:,}")
print(f"Duplicate rows: {profile['duplicates']:,}")
pd.DataFrame(profile['column_profile'])

Rows: 150
Columns: 15
Missing cells: 0
Duplicate rows: 0


,name,dtype,missing,missing_pct,unique,sample
0,Order_ID,str,0,0.0,150,"[ORD-0001, ORD-0002, ORD-0003]"
1,Order_Date,datetime64[us],0,0.0,118,"[2026-02-27 00:00:00, 2026-02-14 00:00:00, 202..."
2,Region,str,0,0.0,4,"[North, East, South]"
3,City,str,0,0.0,12,"[Chandigarh, Kolkata, Hyderabad]"
4,Product,str,0,0.0,7,"[Keyboard, Laptop, Office Chair]"
5,Category,str,0,0.0,3,"[Accessories, Electronics, Furniture]"
6,Customer_Segment,str,0,0.0,3,"[Consumer, Consumer, Enterprise]"
7,Channel,str,0,0.0,3,"[Online, Online, Partner]"
8,Units,int64,0,0.0,22,"[3, 4, 14]"
9,Unit_Price,int64,0,0.0,6,"[2500, 65000, 12000]"


In [4]:
QUESTION = 'What are the most important trends in this data?'
result = run_analysis(source_df, QUESTION, OUTPUT_DIR)

print('Executive summary:')
print(result['summary'])
print('\nMetrics:')
display(pd.DataFrame(result['metrics']))
print('\nKey findings:')
for insight in result['insights']:
    print('-', insight)
print('\nRecommendations:')
for recommendation in result['recommendations']:
    print('-', recommendation)
if result['chart_url']:
    print(f"Chart created at: {ROOT / result['chart_url'].lstrip('/')}")

Executive summary:
Analysis of 150 rows and 15 columns. Units totals 1,346.00 across 150 populated records, with an average of 8.97. Monthly Units increased 39.7% from the first to last observed month.

Metrics:


,label,value
0,Total Units,"1,346.00"
1,Average Units,8.97
2,Range,1.00 – 25.00



Key findings:
- Units totals 1,346.00 across 150 populated records, with an average of 8.97.
- Monthly Units increased 39.7% from the first to last observed month.

Recommendations:
- Validate missing values before using the affected fields in operational decisions.
- Use the displayed trend and summary metrics to investigate the largest changes by segment.
Chart created at: D:\data-analyst-agent\outputs\trend.html


## Edge cases

The following tests target expected real-world CSV conditions: empty input, only categorical fields, missing values and duplicate rows, non-UTF-8 text, and a date-based numeric trend.

In [5]:
import tempfile

cases = {
    'empty_dataframe': pd.DataFrame(columns=['date', 'revenue']),
    'categorical_only': pd.DataFrame({'region': ['North', 'South', 'North'], 'status': ['new', 'new', 'closed']}),
    'missing_and_duplicates': pd.DataFrame({'date': ['2026-01-01', '2026-01-01', None], 'revenue': [100, 100, None]}),
    'date_trend': pd.DataFrame({'date': ['2026-01-01', '2026-02-01', '2026-03-01'], 'revenue': [100, 125, 150]}),
}

checks = []
for name, frame in cases.items():
    case_result = run_analysis(frame, 'Summarize revenue trends', OUTPUT_DIR)
    case_profile = case_result['profile']
    checks.append({
        'case': name,
        'rows': case_profile['rows'],
        'duplicates': case_profile['duplicates'],
        'missing_cells': case_profile['missing_cells'],
        'metric_count': len(case_result['metrics']),
        'chart_created': bool(case_result['chart_url']),
    })

edge_results = pd.DataFrame(checks)
display(edge_results)
assert edge_results.loc[edge_results.case == 'empty_dataframe', 'rows'].item() == 0
assert edge_results.loc[edge_results.case == 'categorical_only', 'metric_count'].item() == 0
assert edge_results.loc[edge_results.case == 'missing_and_duplicates', 'duplicates'].item() == 1
assert edge_results.loc[edge_results.case == 'date_trend', 'chart_created'].item()
print('All in-memory edge-case assertions passed.')

,case,rows,duplicates,missing_cells,metric_count,chart_created
0,empty_dataframe,0,0,0,0,False
1,categorical_only,3,0,0,0,False
2,missing_and_duplicates,3,1,2,3,False
3,date_trend,3,0,0,3,True


All in-memory edge-case assertions passed.


In [6]:
# Verify that CSV reader supports a common legacy encoding.
with tempfile.TemporaryDirectory() as temp_dir:
    latin1_csv = Path(temp_dir) / 'latin1.csv'
    latin1_csv.write_bytes('city,revenue\nMontréal,42\n'.encode('latin-1'))
    decoded = load_csv(latin1_csv)

assert decoded.loc[0, 'city'] == 'Montréal'
assert decoded.loc[0, 'revenue'] == 42
print('Legacy CSV encoding fallback passed.')

Legacy CSV encoding fallback passed.


## Optional API smoke test

Start the API in a terminal with `uv run uvicorn data_analyst.api:app --reload`, then use the web interface to upload CSV, Excel, JSON, or Parquet data. Excel workbooks default to their first worksheet; API clients can supply a `sheet_name` form field to select another worksheet.